In [1]:
!pip install torch safetensors


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from safetensors.torch import save_file, load_file

# Define paths for decoupling model weights from loop metadata
CHECKPOINT_WEIGHTS = "train_checkpoint_test.safetensors"
CHECKPOINT_METADATA = "train_checkpoint_test_metadata.json"

# ---------------------------------------------------------
# Step 1: Define a Simple Architecture & Pipeline States
# ---------------------------------------------------------
class EvaluationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 1)
        
    def forward(self, x):
        return self.linear(x)

def save_checkpoint(model, optimizer, epoch, current_loss):
    """Saves a clean state snapshot using SafeTensors for weights and JSON for metadata."""
    # 1. Extract and save the model parameters (weights/biases) safely
    save_file(model.state_dict(), CHECKPOINT_WEIGHTS)
    
    # 2. Extract and save the loop mechanics and optimizer states
    metadata = {
        "epoch": epoch,
        "loss": float(current_loss),
        # FIX: Remove the inner ["state_dict"] lookup
        "optimizer_state": optimizer.state_dict() if hasattr(optimizer, "state_dict") else {}
    }
    with open(CHECKPOINT_METADATA, "w") as f:
        json.dump(metadata, f, indent=2)
        
    print(f"💾 Checkpoint safely persisted at Epoch {epoch} (Loss: {current_loss:.4f})")

def load_checkpoint(model, optimizer=None):
    """Detects existing checkpoints to dynamically reconstruct the training state."""
    if not os.path.exists(CHECKPOINT_WEIGHTS) or not os.path.exists(CHECKPOINT_METADATA):
        print("🌱 No historical checkpoints found. Initializing a clean training run.")
        return 0 # Start at Epoch 0
    
    print("⚡ Existing checkpoint identified! Initializing state reconciliation...")
    
    # 1. Hydrate the model parameters back via zero-copy mmap pointers
    weights = load_file(CHECKPOINT_WEIGHTS)
    model.load_state_dict(weights)
    
    # 2. Reconstruct the metadata state
    with open(CHECKPOINT_METADATA, "r") as f:
        metadata = json.load(f)
        
    resume_epoch = metadata["epoch"] + 1 # Resume on the *next* epoch
    print(f"🚀 State successfully restored. Resuming automatically from Epoch {resume_epoch}.")
    return resume_epoch

# ---------------------------------------------------------
# Step 2: The Core Training Engine Execution Loop
# ---------------------------------------------------------
def execute_training_pipeline(simulate_crash=False):
    # Initialize basic components
    model = EvaluationModel()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    # Generate static synthetic data for verification
    X = torch.randn(100, 10)
    Y = torch.randn(100, 1)
    
    # Dynamically determine where to start based on local storage context
    start_epoch = load_checkpoint(model, optimizer)
    total_target_epochs = 6
    
    print(f"\n--- Starting Training Loop from Epoch {start_epoch} ---")
    for epoch in range(start_epoch, total_target_epochs):
        # Zero out gradients, run forward pass, calculate loss, backpropagate
        optimizer.zero_grad()
        predictions = model(X)
        loss = criterion(predictions, Y)
        loss.backward()
        optimizer.step()
        
        print(f"Epoch {epoch} Metrics -> Loss: {loss.item():.4f}")
        
        # Save a checkpoint snapshot at the end of every epoch
        save_checkpoint(model, optimizer, epoch, loss.item())
        
        # Intercept and terminate if simulating a hardware failure or process kill
        if simulate_crash and epoch == 2:
            print("\n❌ !!! CRITICAL SYSTEM CRASH SIMULATED !!! ❌")
            print("Process terminated abruptly. VRAM flushed, CPU state wiped.")
            return False
            
    print("\n🎉 Training pipeline completed successfully!")
    return True

# ---------------------------------------------------------
# Step 3: Orchestrating the Lifecycle Demo
# ---------------------------------------------------------
if __name__ == "__main__":
    # Ensure a completely clean directory state for the demo run
    for path in [CHECKPOINT_WEIGHTS, CHECKPOINT_METADATA]:
        if os.path.exists(path):
            os.remove(path)

    print("====================================================")
    print("RUN 1: Executing training loop with simulated crash")
    print("====================================================")
    execute_training_pipeline(simulate_crash=True)
    
    print("\n" + "="*52)
    print("RUN 2: Restarting pipeline (Should heal and resume)")
    print("====================================================")
    success = execute_training_pipeline(simulate_crash=False)
    
    # Final cleanup of artifacts
    # if success:
        # for path in [CHECKPOINT_WEIGHTS, CHECKPOINT_METADATA]:
            # if os.path.exists(path):
                # os.remove(path)

RUN 1: Executing training loop with simulated crash
🌱 No historical checkpoints found. Initializing a clean training run.

--- Starting Training Loop from Epoch 0 ---
Epoch 0 Metrics -> Loss: 0.9468
💾 Checkpoint safely persisted at Epoch 0 (Loss: 0.9468)
Epoch 1 Metrics -> Loss: 0.9347
💾 Checkpoint safely persisted at Epoch 1 (Loss: 0.9347)
Epoch 2 Metrics -> Loss: 0.9232
💾 Checkpoint safely persisted at Epoch 2 (Loss: 0.9232)

❌ !!! CRITICAL SYSTEM CRASH SIMULATED !!! ❌
Process terminated abruptly. VRAM flushed, CPU state wiped.

RUN 2: Restarting pipeline (Should heal and resume)
⚡ Existing checkpoint identified! Initializing state reconciliation...
🚀 State successfully restored. Resuming automatically from Epoch 3.

--- Starting Training Loop from Epoch 3 ---
Epoch 3 Metrics -> Loss: 1.2846
💾 Checkpoint safely persisted at Epoch 3 (Loss: 1.2846)
Epoch 4 Metrics -> Loss: 1.2695
💾 Checkpoint safely persisted at Epoch 4 (Loss: 1.2695)
Epoch 5 Metrics -> Loss: 1.2548
💾 Checkpoint safely